# Clase 04 — Práctica: Hadoop, Hive, Spark y minería de datos

En la clase de teoría vimos qué problema resuelven Hadoop, HDFS, Hive y Spark, el proceso de minería de datos en 6 pasos, y programamos un MapReduce de juguete para contar palabras. Ahora te toca completar tú el código.

## Ejercicio 1: MapReduce con filtro de stopwords y reparto real en nodos

Vas a mejorar el MapReduce de la clase: hay palabras muy frecuentes ("el", "la", "de"...) que no aportan información — se llaman **stopwords** — y de momento el conteo las incluye, ensuciando el resultado. Además, la función que reparte los documentos entre "nodos" todavía no está implementada.

**Tu tarea**:
1. Completa `map_palabras` para que **no** incluya en el resultado las palabras que están en `STOPWORDS`.
2. Completa `repartir_en_nodos` para que reparta la lista `documentos` en `n_nodos` sublistas usando slicing (como vimos en clase).

Pista para el paso 2: `documentos[i:i + tamano_trozo]` recorriendo `i` en saltos de `tamano_trozo`, igual que hicimos en la demo de clase — pero ahora tienes que escribirlo tú dentro de una función reutilizable.

In [ ]:
documentos = [
    "el gato duerme en la alfombra todo el dia",
    "los perros ladran cuando pasa el cartero por la calle",
    "un gato y un perro pueden ser buenos amigos si los presentan bien",
    "la alfombra del salon es el lugar favorito del gato",
    "el cartero trae paquetes todos los dias menos el domingo",
    "los perros y los gatos duermen mucho mas que los humanos",
]

STOPWORDS = {"el", "la", "los", "las", "de", "en", "un", "una",
             "y", "que", "del", "por", "si", "todo", "todos"}


def map_palabras(texto):
    """Fase MAP: devuelve pares (palabra, 1), EXCLUYENDO las
    palabras que están en STOPWORDS."""
    palabras = texto.split()
    pares = []
    for palabra in palabras:
        # TODO: solo añade el par (palabra, 1) si `palabra` NO está en STOPWORDS
        pares.append((palabra, 1))
    return pares


def reduce_conteos(pares):
    """Fase REDUCE: ya está completa, no hace falta tocarla."""
    conteos = {}
    for palabra, valor in pares:
        conteos[palabra] = conteos.get(palabra, 0) + valor
    return conteos


def repartir_en_nodos(lista_documentos, n_nodos):
    """Reparte lista_documentos en n_nodos sublistas casi iguales,
    simulando cómo HDFS repartiría los datos entre nodos."""
    # TODO: calcula tamano_trozo y devuelve la lista de sublistas
    # usando slicing, tal como se hizo en la demo de clase.
    return [lista_documentos]  # placeholder: de momento, "1 solo nodo" con todo


# Con el placeholder de arriba, esto ya funciona (aunque el resultado
# todavía no es el correcto porque los TODO no están resueltos):
nodos = repartir_en_nodos(documentos, 3)
print(f"Número de nodos simulados: {len(nodos)}")

pares_intermedios = []
for datos_nodo in nodos:
    for doc in datos_nodo:
        pares_intermedios.extend(map_palabras(doc))

resultado = reduce_conteos(pares_intermedios)
top_5 = sorted(resultado.items(), key=lambda par: par[1], reverse=True)[:5]
print("Top 5 (antes de resolver los TODO):", top_5)

<details>
<summary>💡 Ver solución</summary>

```python
documentos = [
    "el gato duerme en la alfombra todo el dia",
    "los perros ladran cuando pasa el cartero por la calle",
    "un gato y un perro pueden ser buenos amigos si los presentan bien",
    "la alfombra del salon es el lugar favorito del gato",
    "el cartero trae paquetes todos los dias menos el domingo",
    "los perros y los gatos duermen mucho mas que los humanos",
]

STOPWORDS = {"el", "la", "los", "las", "de", "en", "un", "una",
             "y", "que", "del", "por", "si", "todo", "todos"}


def map_palabras(texto):
    palabras = texto.split()
    pares = []
    for palabra in palabras:
        if palabra not in STOPWORDS:  # <- filtro de stopwords
            pares.append((palabra, 1))
    return pares


def reduce_conteos(pares):
    conteos = {}
    for palabra, valor in pares:
        conteos[palabra] = conteos.get(palabra, 0) + valor
    return conteos


def repartir_en_nodos(lista_documentos, n_nodos):
    tamano_trozo = len(lista_documentos) // n_nodos + 1
    return [
        lista_documentos[i:i + tamano_trozo]
        for i in range(0, len(lista_documentos), tamano_trozo)
    ]


nodos = repartir_en_nodos(documentos, 3)
print(f"Número de nodos simulados: {len(nodos)}")

pares_intermedios = []
for datos_nodo in nodos:
    for doc in datos_nodo:
        pares_intermedios.extend(map_palabras(doc))

resultado = reduce_conteos(pares_intermedios)
top_5 = sorted(resultado.items(), key=lambda par: par[1], reverse=True)[:5]
print("Top 5 (ya sin stopwords):", top_5)
```
</details>

## Ejercicio 2: identificar los 6 pasos de la minería de datos

Una empresa de retail quiere reducir la cancelación de clientes (churn). A continuación tienes una lista de actividades que su equipo de datos fue realizando, **en orden**, pero sin etiquetar con el paso del proceso de minería de datos al que corresponde.

**Tu tarea**: completa la función `identificar_paso`, que recibe la descripción de una actividad (en minúsculas) y devuelve el número de paso (1 a 6) al que corresponde, usando palabras clave. Ya te dejamos resueltos los pasos 1 y 6 como ejemplo — completa el resto siguiendo la misma lógica.

In [ ]:
actividades = [
    "definimos que queremos predecir que clientes van a cancelar su suscripcion",
    "reunimos datos de facturacion, soporte tecnico y uso de la app",
    "revisamos los datos y encontramos columnas con valores imposibles y duplicados",
    "convertimos las fechas a un formato unico y normalizamos las monedas",
    "entrenamos un arbol de decision para predecir la cancelacion",
    "medimos la precision del modelo y presentamos los resultados al equipo directivo",
]

PASOS = {
    1: "Fijar el objetivo",
    2: "Seleccionar fuentes de datos",
    3: "Preprocesar",
    4: "Transformar",
    5: "Minar",
    6: "Evaluar",
}


def identificar_paso(descripcion):
    """Devuelve el número de paso (1-6) según palabras clave
    presentes en la descripción de la actividad."""
    if "definimos" in descripcion or "queremos predecir" in descripcion:
        return 1
    # TODO: paso 2 — palabras clave como "reunimos" o "datos de"
    # TODO: paso 3 — palabras clave como "revisamos", "duplicados", "valores imposibles"
    # TODO: paso 4 — palabras clave como "convertimos", "normalizamos", "formato"
    # TODO: paso 5 — palabras clave como "entrenamos", "modelo", "arbol"
    if "medimos" in descripcion or "presentamos" in descripcion:
        return 6
    return 0  # sin clasificar todavía


for actividad in actividades:
    paso = identificar_paso(actividad)
    nombre_paso = PASOS.get(paso, "SIN CLASIFICAR")
    print(f"[{nombre_paso}] {actividad}")

<details>
<summary>💡 Ver solución</summary>

```python
def identificar_paso(descripcion):
    if "definimos" in descripcion or "queremos predecir" in descripcion:
        return 1
    elif "reunimos" in descripcion or "datos de" in descripcion:
        return 2
    elif "revisamos" in descripcion or "duplicados" in descripcion or "valores imposibles" in descripcion:
        return 3
    elif "convertimos" in descripcion or "normalizamos" in descripcion or "formato" in descripcion:
        return 4
    elif "medimos" in descripcion or "presentamos" in descripcion:
        return 6
    elif "entrenamos" in descripcion or "arbol" in descripcion:
        return 5
    return 0


for actividad in actividades:
    paso = identificar_paso(actividad)
    nombre_paso = PASOS.get(paso, "SIN CLASIFICAR")
    print(f"[{nombre_paso}] {actividad}")
```
</details>

## Para volver

¿Quieres repasar la teoría antes de seguir? Vuelve al notebook de clase: [clase04_hadoop_hive_spark.ipynb](../notebooks_clase/clase04_hadoop_hive_spark.ipynb)